# UD7.04. Cuándo compensa automatizar

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloques 5 y 7 de los apuntes · Criterios **4.c** y **4.d**

---

Los tres cuadernos anteriores calculan lo que se ahorra. Falta lo que se gasta, que es
donde mueren la mayoría de los proyectos: **no en la construcción, en el mantenimiento**.

Y falta la parte que hace que esto sirva para algo más que para un caso: aplicar la misma
cuenta a **los once procesos manuales de TechStore** y comprobar que la respuesta no es la
misma en todos. Ése es el material de partida de PR7.

Al final tienes que poder contestar:

1. ¿En cuánto tiempo se paga la automatización del buzón, y de qué depende ese plazo?
2. ¿Qué variable decide de verdad: la tecnología, el volumen o el coste del error?
3. De los once procesos de TechStore, ¿cuáles compensan y cuáles no?
4. ¿Cuál **no se debe automatizar** aunque los números salgan, y por qué el argumento no
   es moral sino numérico?

In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", None)

BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD7/datos/")
os.makedirs("datos", exist_ok=True)
for fichero in ["buzon_techstore.csv", "procesos_techstore.csv"]:
    ruta = os.path.join("datos", fichero)
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(BASE + fichero, ruta)

procesos = pd.read_csv(os.path.join("datos", "procesos_techstore.csv"))
print(procesos.to_string(index=False))

## 1. Los supuestos, todos declarados

Nada de lo que viene es un dato de la contabilidad de TechStore. Son **supuestos de
trabajo**, y la norma de la unidad es que se escriben con nombre antes de usarlos,
porque todo lo demás depende de ellos.

> **Un dictamen con cifras y sin supuestos declarados no vale nada**, y quien lo lea no
> puede discutirlo. Con los supuestos escritos, cualquiera puede cambiar uno y ver qué
> pasa, que es exactamente lo que vamos a hacer en el apartado 3.

In [ ]:
COSTE_HORA = 18.50            # coste empresa de una hora de la persona del proceso
COSTE_HORA_DESARROLLO = 32.00  # coste empresa de una hora de quien lo construye

# Del cuaderno UD7_03, con umbral 0,70:
COSTE_MANUAL_BUZON = 18_805    # EUR/año, tiempo + errores
COSTE_AUTOMATICO_BUZON = 8_645  # EUR/año, con umbral 0,70

HORAS_CONSTRUCCION = 120       # dos personas, tres semanas, contando pruebas
HORAS_MANTENIMIENTO_MES = 4    # reentrenar, vigilar la deriva, arreglar lo que cambie

construccion = HORAS_CONSTRUCCION * COSTE_HORA_DESARROLLO
mantenimiento_ano = HORAS_MANTENIMIENTO_MES * 12 * COSTE_HORA_DESARROLLO

print(f"Construcción ({HORAS_CONSTRUCCION} h a {COSTE_HORA_DESARROLLO:.0f} EUR/h): "
      f"{construccion:>8,.0f} EUR, una sola vez")
print(f"Mantenimiento ({HORAS_MANTENIMIENTO_MES} h/mes):            "
      f"{mantenimiento_ano:>8,.0f} EUR al año")

Las 120 horas de construcción no son solo programar. Conviene desglosarlas, porque el
desglose es lo que separa un presupuesto de una corazonada:

| Partida | Horas | Por qué |
|---|---|---|
| Entender el proceso y medirlo | 16 | Los cuadernos `UD7_01` y `UD7_02` |
| **Auditar una muestra del histórico** | **20** | Bloque 10.2: el modelo hereda los errores de quien etiquetó |
| Preparar los datos y el contexto | 20 | Conectar el sistema de pedidos, que es de donde salen los días |
| Entrenar, medir y elegir el umbral | 16 | El cuaderno `UD7_03` |
| Integrarlo con el buzón real | 28 | Casi siempre lo más largo, y casi siempre lo peor estimado |
| Pruebas, puesta en marcha y documentación | 20 | Incluye el registro de decisiones que exige el artículo 22 del RGPD |

Las 20 horas de auditoría del histórico son las que nadie presupuesta y las que el
cuaderno `UD7_02` demostró que hacen falta.

## 2. El punto de equilibrio del buzón

In [ ]:
ahorro_bruto = COSTE_MANUAL_BUZON - COSTE_AUTOMATICO_BUZON
ahorro_neto = ahorro_bruto - mantenimiento_ano
meses = construccion / (ahorro_neto / 12)

print(f"Coste del proceso a mano        {COSTE_MANUAL_BUZON:>9,.0f} EUR/año")
print(f"Coste con triaje automático     {COSTE_AUTOMATICO_BUZON:>9,.0f} EUR/año")
print(f"Ahorro bruto                    {ahorro_bruto:>9,.0f} EUR/año")
print(f"Menos mantenimiento             {-mantenimiento_ano:>9,.0f} EUR/año")
print(f"Ahorro neto                     {ahorro_neto:>9,.0f} EUR/año")
print()
print(f"Construcción                    {construccion:>9,.0f} EUR, una vez")
print(f"PUNTO DE EQUILIBRIO             {meses:>9.1f} meses")

## 3. La sensibilidad, que es lo que de verdad se entrega

Un punto de equilibrio solo es una cifra. Lo que se evalúa en PR7 es **saber de qué
depende**, y eso se enseña moviendo un supuesto cada vez.

In [ ]:
def equilibrio(ahorro_bruto_ano, mantenimiento, coste_construccion):
    neto = ahorro_bruto_ano - mantenimiento
    return np.inf if neto <= 0 else coste_construccion / (neto / 12)


print("VOLUMEN  (el ahorro es proporcional al número de casos)")
print(f"  {'volumen':>10} {'ahorro neto':>14} {'equilibrio':>14}")
for factor in [0.25, 0.5, 1.0, 2.0, 4.0]:
    neto = ahorro_bruto * factor - mantenimiento_ano
    m = equilibrio(ahorro_bruto * factor, mantenimiento_ano, construccion)
    plazo = "nunca" if np.isinf(m) else f"{m:.1f} meses"
    print(f"  {'x' + format(factor, 'g'):>10} {neto:>10,.0f} EUR {plazo:>14}")

print("\nMANTENIMIENTO  (el supuesto que más se subestima)")
print(f"  {'h/mes':>10} {'coste/año':>14} {'equilibrio':>14}")
for horas in [2, 4, 8, 12, 16]:
    mant = horas * 12 * COSTE_HORA_DESARROLLO
    m = equilibrio(ahorro_bruto, mant, construccion)
    plazo = "nunca" if np.isinf(m) else f"{m:.1f} meses"
    print(f"  {horas:>10} {mant:>10,.0f} EUR {plazo:>14}")

print("\nCOSTE DEL ERROR  (si la garantía costara menos de lo que hemos supuesto)")
print(f"  {'factor':>10} {'ahorro bruto':>15} {'equilibrio':>14}")
for factor in [0.25, 0.5, 1.0, 2.0]:
    # Solo se escala la parte del ahorro que viene de los errores. Del cuaderno UD7_03:
    ahorro_errores = 12_518 - 4_332
    ahorro_tiempo = ahorro_bruto - ahorro_errores
    bruto = ahorro_tiempo + ahorro_errores * factor
    m = equilibrio(bruto, mantenimiento_ano, construccion)
    plazo = "nunca" if np.isinf(m) else f"{m:.1f} meses"
    print(f"  {'x' + format(factor, 'g'):>10} {bruto:>11,.0f} EUR {plazo:>14}")

Tres conclusiones, y las tres van en el dictamen:

1. **El volumen es la variable, no la tecnología.** El mismo flujo exactamente, con la
   cuarta parte de los mensajes, tarda casi cuatro años en pagarse. Para una empresa
   pequeña eso es lo mismo que no pagarse: en cuatro años han cambiado el catálogo, el
   proveedor de correo y la mitad del equipo.
2. **El mantenimiento decide más que la construcción.** Pasar de 4 a 16 horas al mes no
   cambia lo que cuesta construirlo y sin embargo puede dejar el proyecto sin retorno.
   Es el supuesto que más se subestima y el que menos se discute.
3. **Si el coste del error fuera la cuarta parte, el proyecto seguiría compensando, pero
   ya no por lo mismo.** Conviene saber qué parte del ahorro es tiempo y qué parte es
   errores, porque son dos argumentos distintos ante dos interlocutores distintos.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))

factores = np.linspace(0.15, 3, 60)
meses_volumen = [equilibrio(ahorro_bruto * f, mantenimiento_ano, construccion) for f in factores]
ejes[0].plot(factores, meses_volumen, color="#4878A8", linewidth=2)
ejes[0].axhline(12, color="#B4453C", linestyle="--", label="un año")
ejes[0].axvline(1.0, color="#3B7A57", linestyle=":", label="volumen actual")
ejes[0].set_ylim(0, 60)
ejes[0].set_xlabel("volumen, respecto del actual")
ejes[0].set_ylabel("meses hasta el equilibrio")
ejes[0].set_title("El volumen decide")
ejes[0].legend()

horas = np.arange(1, 25)
meses_mant = [equilibrio(ahorro_bruto, h * 12 * COSTE_HORA_DESARROLLO, construccion)
              for h in horas]
ejes[1].plot(horas, meses_mant, color="#4878A8", linewidth=2)
ejes[1].axhline(12, color="#B4453C", linestyle="--", label="un año")
ejes[1].axvline(HORAS_MANTENIMIENTO_MES, color="#3B7A57", linestyle=":",
                label="supuesto: 4 h/mes")
ejes[1].set_ylim(0, 60)
ejes[1].set_xlabel("horas de mantenimiento al mes")
ejes[1].set_ylabel("meses hasta el equilibrio")
ejes[1].set_title("El mantenimiento también")
ejes[1].legend()

plt.tight_layout()
plt.show()

## 4. Los once procesos de TechStore

Y ahora lo que hace que esto sirva más allá del buzón. El catálogo trae once procesos
manuales de la empresa con su volumen, su tiempo por caso, su tasa de error y lo que
cuesta equivocarse. Vamos a aplicarles la misma cuenta a los once.

Para eso hacen falta tres supuestos más, y también van declarados. Los tres dependen de
**la variabilidad del proceso**, que es lo que decide cuánto se puede automatizar de
verdad:

| Variabilidad | Parte del tiempo que se automatiza | Parte de los errores que se evita | Mantenimiento |
|---|---|---|---|
| Baja | 80 % | 70 % | 2 h/mes |
| Media | 55 % | 50 % | 4 h/mes |
| Alta | 35 % | 30 % | 6 h/mes |

Y la construcción: 40 horas de base, más 40 si la variabilidad es alta, más 30 si los
datos solo están parciales.

In [ ]:
AUTOMATIZA_TIEMPO = {"baja": 0.80, "media": 0.55, "alta": 0.35}
EVITA_ERRORES = {"baja": 0.70, "media": 0.50, "alta": 0.30}
MANTENIMIENTO_H_MES = {"baja": 2, "media": 4, "alta": 6}


def analiza(fila):
    casos_ano = fila["casos_mes"] * 12
    horas_ano = casos_ano * fila["minutos_caso"] / 60
    coste_tiempo = horas_ano * COSTE_HORA
    coste_errores = casos_ano * fila["tasa_error_manual"] * fila["coste_error_euros"]

    variabilidad = fila["variabilidad"]
    ahorro = (coste_tiempo * AUTOMATIZA_TIEMPO[variabilidad]
              + coste_errores * EVITA_ERRORES[variabilidad])
    horas_construccion = (40
                          + (40 if variabilidad == "alta" else 0)
                          + (30 if fila["datos_disponibles"] == "parciales" else 0))
    coste_construccion = horas_construccion * COSTE_HORA_DESARROLLO
    mantenimiento = MANTENIMIENTO_H_MES[variabilidad] * 12 * COSTE_HORA_DESARROLLO
    neto = ahorro - mantenimiento

    return pd.Series({
        "coste_manual": coste_tiempo + coste_errores,
        "de eso, errores": coste_errores,
        "ahorro_bruto": ahorro,
        "mantenimiento": mantenimiento,
        "ahorro_neto": neto,
        "construccion": coste_construccion,
        "meses": np.inf if neto <= 0 else coste_construccion / (neto / 12),
    })


analisis = pd.concat([procesos, procesos.apply(analiza, axis=1)], axis=1)
salida = analisis[["proceso_id", "proceso", "variabilidad", "coste_manual",
                   "de eso, errores", "ahorro_neto", "construccion", "meses",
                   "decide_sobre_personas"]].copy()
salida = salida.sort_values("meses")
for columna in ["coste_manual", "de eso, errores", "ahorro_neto", "construccion"]:
    salida[columna] = salida[columna].round(0)
salida["meses"] = salida["meses"].round(1)
print(salida.to_string(index=False))

Léela despacio, porque ahí está el ejercicio entero de PR7.

In [ ]:
print("Lo que el orden de esa tabla enseña:\n")
mejor = salida.iloc[0]
peor = salida[np.isfinite(salida["meses"])].iloc[-1]
print(f"  El que antes se paga:  {mejor['proceso_id']} {mejor['proceso']}")
print(f"                         {mejor['meses']:.1f} meses")
print(f"  El que más tarda:      {peor['proceso_id']} {peor['proceso']}")
print(f"                         {peor['meses']:.1f} meses")
print()
nunca = salida[~np.isfinite(salida["meses"])]
if len(nunca):
    print("  No se pagan nunca (el mantenimiento se come el ahorro):")
    for _, f in nunca.iterrows():
        print(f"    {f['proceso_id']} {f['proceso']}")
print()
print("  Y el dato que más sorprende: la proporción del coste que son ERRORES")
proporcion = (analisis["de eso, errores"] / analisis["coste_manual"]).sort_values(ascending=False)
for i in proporcion.head(4).index:
    print(f"    {analisis.loc[i, 'proceso_id']} {analisis.loc[i, 'proceso'][:44]:<44} "
          f"{proporcion[i]:>5.0%}")

> **La respuesta no es la misma en los once, y ésa es toda la lección.** Hay procesos con
> mucho volumen y tarea trivial que se pagan en semanas, procesos con poco volumen y
> tarea cara que no se pagan nunca, y procesos donde casi todo el coste son errores y por
> tanto el argumento no es el tiempo ahorrado sino la calidad.
>
> Quien conteste «hay que automatizar los procesos repetitivos» no ha hecho la cuenta.

## 5. Un momento: el buzón sale en esa tabla, y sale mal

Si has mirado la tabla con atención habrás visto algo raro. **PR-01 es el buzón**, el
mismo proceso que en el apartado 2 se pagaba en 5,3 meses, y en la tabla de los once
aparece entre los que **no se pagan nunca**.

No es una errata. Es lo más útil que enseña este cuaderno, así que vamos a mirarlo de
frente.

In [ ]:
fila_catalogo = procesos[procesos["proceso_id"] == "PR-01"].iloc[0]

# Lo que el catálogo dice del buzón frente a lo que salió al medirlo en UD7_01.
buzon = pd.read_csv(os.path.join("datos", "buzon_techstore.csv"))
minutos_medido = buzon["minutos_manual"].mean()
error_medido = float((buzon["categoria"] != buzon["categoria_asignada"]).mean())
coste_error_medido = 12_518 / int((buzon["categoria"] != buzon["categoria_asignada"]).sum())

print(f"{'':<26} {'catálogo':>12} {'medido':>12}")
print(f"{'casos al mes':<26} {fila_catalogo['casos_mes']:>12} {len(buzon) / 12:>12.0f}")
print(f"{'minutos por caso':<26} {fila_catalogo['minutos_caso']:>12.1f} {minutos_medido:>12.1f}")
print(f"{'tasa de error':<26} {fila_catalogo['tasa_error_manual']:>11.1%} {error_medido:>11.1%}")
print(f"{'coste medio del error':<26} {fila_catalogo['coste_error_euros']:>11.2f}€ "
      f"{coste_error_medido:>10.2f}€")
print()
coste_catalogo = (fila_catalogo["casos_mes"] * 12 * fila_catalogo["minutos_caso"] / 60 * COSTE_HORA
                  + fila_catalogo["casos_mes"] * 12 * fila_catalogo["tasa_error_manual"]
                  * fila_catalogo["coste_error_euros"])
print(f"Coste anual según el catálogo: {coste_catalogo:>9,.0f} EUR")
print(f"Coste anual medido:            {COSTE_MANUAL_BUZON:>9,.0f} EUR")
print(f"El catálogo se queda corto por un factor de "
      f"{COSTE_MANUAL_BUZON / coste_catalogo:.1f}.")

El catálogo no miente por malicia: son las cifras que la empresa **cree** que tiene,
estimadas de memoria por quien lleva el área. Y están mal en las cuatro columnas, sobre
todo en dos:

- **La tasa de error**: se creía un 8 %, es un 13,2 %. Nadie la había medido nunca,
  porque para medirla hay que revisar una muestra a mano.
- **El coste medio del error**: se creía 9 euros, son casi 40, porque **nadie había
  separado los errores caros de los baratos**. Es exactamente lo que el cuaderno `UD7_03`
  hizo con la matriz de costes.

Con las cifras del catálogo, el buzón es un proceso que **no merece la pena tocar**. Con
las cifras medidas, se paga en cinco meses. **La decisión se invierte entera, y en medio
no hay ninguna tecnología: hay dos días de medir.**

> **La primera tarea de un proyecto de automatización no es elegir herramienta: es
> comprobar los números con los que te han encargado el proyecto.** Y suele ser también
> la más rentable, porque a veces los tira y a veces, como aquí, los convierte en un
> proyecto que sí vale la pena.
>
> Los otros diez procesos del catálogo llevan cifras estimadas igual que ésta. **No hay
> ninguna razón para pensar que son mejores**, y en PR7 se espera que digas qué harías
> para comprobar las del tuyo y cuánto costaría comprobarlas.

## 6. El proceso que no se debe automatizar

La columna `decide_sobre_personas` no entra en ninguna de las cuentas anteriores, y sin
embargo manda sobre todas ellas.

In [ ]:
sobre_personas = analisis[analisis["decide_sobre_personas"] == "sí"].copy()
sobre_personas["meses"] = sobre_personas["meses"].round(1)
print(sobre_personas[["proceso_id", "proceso", "casos_mes", "meses", "reversible"]]
      .to_string(index=False))
print()
print("De los tres, el que hay que mirar con lupa es PR-10.")
pr10 = analisis[analisis["proceso_id"] == "PR-10"].iloc[0]
print(f"  {pr10['proceso']}")
print(f"  {pr10['casos_mes']} casos al mes, {pr10['minutos_caso']:.0f} min cada uno")
print(f"  coste manual: {pr10['coste_manual']:,.0f} EUR/año")
print(f"  ahorro neto:  {pr10['ahorro_neto']:,.0f} EUR/año")
print(f"  equilibrio:   {pr10['meses']:.1f} meses" if np.isfinite(pr10["meses"])
      else "  equilibrio:   nunca")

**PR-10 es la criba de currículums para el almacén de campaña.** Es un sistema de **alto
riesgo del anexo III** del Reglamento (UE) 2024/1689: sistemas destinados a la selección
de personal. Eso implica gestión de riesgos, gobernanza de datos, documentación técnica,
registro de actividad, **supervisión humana efectiva** y evaluación de la conformidad.
Además, el artículo 64.4.d) del Estatuto de los Trabajadores obliga a informar a la
representación de los trabajadores de los parámetros del algoritmo, y el artículo 22 del
RGPD entra en juego si la criba decide por sí sola.

Vamos a ponerle precio a eso, que es lo que pide el criterio 4.d.

In [ ]:
# Estimación prudente del coste de cumplimiento, en horas, la primera vez y cada año.
CUMPLIMIENTO = {
    "Evaluación de impacto y gestión de riesgos": (40, 8),
    "Documentación técnica y registro de actividad": (30, 6),
    "Gobernanza de datos y control de sesgos": (35, 12),
    "Información a la representación de los trabajadores": (8, 2),
    "Supervisión humana: revisión de cada decisión": (0, 9 * 12 * 10 / 60),
}
print(f"{'partida':<52} {'1ª vez':>8} {'al año':>8}")
primera, anual = 0, 0
for partida, (h_inicial, h_anual) in CUMPLIMIENTO.items():
    primera += h_inicial
    anual += h_anual
    print(f"{partida:<52} {h_inicial:>7.0f}h {h_anual:>7.1f}h")
print(f"{'TOTAL':<52} {primera:>7.0f}h {anual:>7.1f}h")
print()
coste_cumplimiento_inicial = primera * COSTE_HORA_DESARROLLO
coste_cumplimiento_anual = anual * COSTE_HORA_DESARROLLO
print(f"Coste de cumplir la ley: {coste_cumplimiento_inicial:,.0f} EUR la primera vez "
      f"y {coste_cumplimiento_anual:,.0f} EUR al año")
print(f"Coste de hacerlo a mano: {pr10['coste_manual']:,.0f} EUR al año")
print()
veces = coste_cumplimiento_inicial / pr10["coste_manual"]
print(f"Solo la puesta en cumplimiento cuesta {veces:.1f} veces lo que cuesta")
print("hacer el proceso entero a mano durante un año.")

> **Ésta es la conclusión que hay que saber escribir, y no es moralista:**
>
> No es que sea ilegal automatizar la criba de currículums. Es que **con nueve casos al
> mes, el coste de cumplir la ley es varias veces el coste de hacerlo a mano**, y encima
> el modelo aprendería a cribar como se cribó antes, con los sesgos de antes —el bloque
> 10.2 lo midió— y esta vez sobre personas que buscan trabajo.
>
> La recomendación es **no automatizarlo**, y el argumento es numérico. Un dictamen que
> dijera «no lo automatizamos porque afecta a personas y hay que tener cuidado» no
> puntúa: no tiene número y no se puede discutir. Uno que ponga estas dos cifras al lado,
> sí.

Y el revés de la moneda, que también hay que saber decir: **si el volumen fuera de nueve
mil currículums al mes en lugar de nueve**, la cuenta cambiaría, el cumplimiento se
prorratearía entre muchos más casos y la conversación sería otra —seguiría siendo alto
riesgo, con todas sus obligaciones, pero ya no se despacharía con una división—.

## 7. Lo que hay que llevarse

In [ ]:
print(f"""
EL BUZÓN DE TECHSTORE, ENTERO
=============================
A mano                          {COSTE_MANUAL_BUZON:>9,.0f} EUR/año
Automático con umbral 0,70      {COSTE_AUTOMATICO_BUZON:>9,.0f} EUR/año
Ahorro bruto                    {ahorro_bruto:>9,.0f} EUR/año
Mantenimiento                   {-mantenimiento_ano:>9,.0f} EUR/año
Ahorro neto                     {ahorro_neto:>9,.0f} EUR/año
Construcción, una vez           {construccion:>9,.0f} EUR
Punto de equilibrio             {meses:>9.1f} meses

Deja de compensar si:
  el volumen baja por debajo de la mitad      -> {equilibrio(ahorro_bruto * 0.5, mantenimiento_ano, construccion):.0f} meses
  el mantenimiento pasa de 16 h al mes        -> {equilibrio(ahorro_bruto, 16 * 12 * COSTE_HORA_DESARROLLO, construccion):.0f} meses
""")

1. **Las cifras que te dan al encargarte el proyecto están mal.** Comprobarlas es lo
   primero, y en el buzón cambia la respuesta de «no compensa» a «cinco meses».
2. **El mantenimiento decide más que la construcción.** Es la partida que nadie discute y
   la que puede dejar un proyecto sin retorno.
3. **Las 20 horas de auditar el histórico no son opcionales**, y son las que nadie
   presupuesta.
4. **El volumen es la variable, no la tecnología.**
5. **La respuesta no es la misma en los once procesos.** «Hay que automatizar lo
   repetitivo» no es una respuesta: es una consigna.
6. **Hay procesos que no se deben automatizar, y el buen argumento es numérico:** el
   coste de cumplir la ley supera el de hacerlo a mano.

---

### Para PR7

Este cuaderno es la plantilla. En PR7 vas a elegir **un proceso del catálogo que no sea
el buzón**, rehacer esta cuenta con **tus** supuestos —los de aquí son discutibles y se
espera que los discutas— y escribir el dictamen de conveniencia del bloque 12 de los
apuntes.

Lo que no vale: copiar los supuestos de este cuaderno sin mirarlos. `AUTOMATIZA_TIEMPO`
y `EVITA_ERRORES` son números puestos a ojo para poder ordenar los once procesos, y en
un proceso concreto hay que justificarlos o sustituirlos por una medición.